# Lab 04.4 — Real-Time Color Detection

## Objective

Build a simple real-time detector for **red, green, blue and yellow** objects using the Logitech C270.

The processing pipeline is:

```text
camera -> BGR -> HSV -> color masks -> contours -> label
```

For reliable classroom experiments, use saturated objects such as colored paper, plastic blocks or markers.

## Camera setup — provided code

Run this helper cell once in this notebook. It uses the previously verified **V4L2 / `/dev/video0` → 20 warm-up frames → capture → inline JPEG display → release** sequence. The requested resolution is 640×480; use `frame.shape` to check the actual resolution.

The `with camera_session()` block releases the camera through `finally`, including on Python errors or a normal Jupyter interrupt. Only one notebook/kernel should use the camera at a time. If Linux assigns a different capture device, change **CAMERA_DEVICE in each notebook you use**; notebook variables are not shared.

The helper is supplied; you do not need to implement it. Time limits are checked between reads: a USB driver call that blocks can delay an interrupt or timeout.

In [ ]:
import cv2
import numpy as np
import time
from contextlib import contextmanager
from IPython.display import display, Image, clear_output

CAMERA_DEVICE = "/dev/video0"
WARMUP_FRAMES = 20

def read_frame(cap):
    ok, frame = cap.read()
    if not ok or frame is None or frame.size == 0:
        raise RuntimeError(
            "No camera frame. Check the USB connection and close other camera notebooks."
        )
    return frame

@contextmanager
def camera_session():
    """Provided helper: open, warm up, and always release the camera."""
    cap = cv2.VideoCapture(CAMERA_DEVICE, cv2.CAP_V4L2)
    try:
        if not cap.isOpened():
            raise RuntimeError(
                "Cannot open " + CAMERA_DEVICE +
                ". Check the device and shut down other kernels using the camera."
            )
        # Request a modest resolution; the driver may negotiate another one.
        cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
        cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
        warmup_start = time.monotonic()
        for _ in range(WARMUP_FRAMES):
            read_frame(cap)  # Discard frames while exposure settles.
            if time.monotonic() - warmup_start > 10:
                raise RuntimeError("Camera warm-up is too slow. Check the USB connection.")
        yield cap
    finally:
        cap.release()
        print("Camera released.")

def show_bgr(frame, replace=False):
    ok, jpg = cv2.imencode(".jpg", frame)
    if not ok:
        raise RuntimeError("JPEG encoding failed.")
    if replace:
        clear_output(wait=True)
    display(Image(data=jpg.tobytes()))


In [ ]:
COLOR_RANGES = {
    "BLUE":   [((100, 100, 60), (130, 255, 255))],
    "GREEN":  [((35, 80, 60), (85, 255, 255))],
    "YELLOW": [((20, 100, 100), (34, 255, 255))],
    "RED":    [((0, 100, 80), (10, 255, 255)),
               ((170, 100, 80), (179, 255, 255))]
}

DRAW_COLORS = {
    "BLUE": (255, 0, 0),
    "GREEN": (0, 255, 0),
    "YELLOW": (0, 255, 255),
    "RED": (0, 0, 255)
}

In [ ]:
def make_mask(hsv, ranges):
    mask = np.zeros(hsv.shape[:2], dtype=np.uint8)

    for lower, upper in ranges:
        part = cv2.inRange(
            hsv,
            np.array(lower, dtype=np.uint8),
            np.array(upper, dtype=np.uint8)
        )
        mask = cv2.bitwise_or(mask, part)

    kernel = np.ones((5, 5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    return mask

## Reading the detector output

Each color mask is processed separately. A contour is the boundary of a selected region; `contourArea()` rejects small regions and `boundingRect()` supplies the box drawn around each candidate. Lab04.5 explains these measurements in more detail. A box means the region matches a color threshold, not that the program knows the object's identity.

## Run the detector

Hold a large, saturated colored object in front of the camera. The session stops after at most 180 processed frames or approximately 15 seconds after warm-up. These are limits checked between reads, not a guaranteed driver timeout. Use **Kernel → Interrupt** to stop early; rerun this cell for another session.

640×480 is requested. If you use another resolution, the fixed `MIN_AREA` threshold may need adjustment. `time.sleep(0.03)` is only a short pause; it does not guarantee a frame rate. Display and processing also take time.

In [ ]:
MIN_AREA = 1200  # Contour area threshold; pixels squared.
MAX_FRAMES = 180
MAX_SECONDS = 15

with camera_session() as cap:
    start = time.monotonic()
    for _ in range(MAX_FRAMES):
        if time.monotonic() - start >= MAX_SECONDS:
            break
        frame = read_frame(cap)

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

        for name, ranges in COLOR_RANGES.items():
            mask = make_mask(hsv, ranges)
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            for cnt in contours:
                area = cv2.contourArea(cnt)
                if area < MIN_AREA:
                    continue

                x, y, w, h = cv2.boundingRect(cnt)

                cv2.rectangle(frame, (x, y), (x+w, y+h), DRAW_COLORS[name], 2)
                cv2.putText(frame, name, (x, max(25, y-8)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7,
                            DRAW_COLORS[name], 2)

        show_bgr(frame, replace=True)

        time.sleep(0.03)



## Short exercise

1. Show a green and a blue object together.
2. Change `MIN_AREA` from 1200 to 400 and rerun the detector; observe whether smaller unwanted regions appear.
3. Restore 1200 and shade one object slightly. Observe whether its box remains stable.

**Record:** one condition that made detection worse. The HSV ranges are examples to tune, not universal color definitions.